In [10]:
import openai, json

client = openai.OpenAI()
messages = []

In [11]:
import requests
import json

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

def get_popular_movies():
    """인기 영화 목록을 반환합니다."""
    resp = requests.get(f"{BASE_URL}/movies")
    resp.raise_for_status()
    return json.dumps(resp.json(), ensure_ascii=False)

def get_movie_details(id):
    """ID로 영화 상세 정보를 조회합니다."""
    resp = requests.get(f"{BASE_URL}/movies/{id}")
    resp.raise_for_status()
    return json.dumps(resp.json(), ensure_ascii=False)

def get_movie_credits(id):
    """영화의 출연진 및 제작진 정보를 반환합니다."""
    resp = requests.get(f"{BASE_URL}/movies/{id}/credits")
    resp.raise_for_status()
    return json.dumps(resp.json(), ensure_ascii=False)

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits
}

In [12]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of popular movies."
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get the details of a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The ID of the movie."
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get the credits of a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The ID of the movie."
                    }
                },
                "required": ["id"]
            }
        }
    }
]

In [13]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role":"assistant", 
                "content":message.content or "",
                "tool_calls":[
                    {
                        "id":tool_call.id,
                        "type":"function",
                        "function":{
                            "name":tool_call.function.name,
                            "arguments":tool_call.function.arguments
                        },
                    } for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)
            
            # print(f"Ran {function_name} with args {arguments} for a result of {result}")
            print(f"Ran {function_name} with args {arguments}")

            messages.append({
                "role":"tool",
                "tool_call_id":tool_call.id,
                "name":function_name,
                "content":result,
            })
        Call_ai()
    else :
        messages.append({"role":"assistant", "content":message.content})
        print(f"AI : {message.content}")



def Call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [14]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role":"user", "content":message})
        print(f"User : {message}")
        Call_ai()

User : 안녕
AI : 안녕하세요! 어떻게 도와드릴까요?
User : 난 SF 영화를 좋아해
AI : SF 영화는 정말 흥미로운 장르죠! 최근에 나온 인기 SF 영화에 대해 알고 싶으신가요? 아니면 특정 영화에 대해 궁금한 점이 있으신가요?
User : 메트릭스랑 arrival은 이미 본 영화야
Calling function: get_popular_movies with {}
Ran get_popular_movies with args {}
AI : 최근에 나온 인기 SF 영화 몇 편을 소개할게요:

1. **Mercy** (2026-01-20)
   - **개요**: 가까운 미래, 한 형사가 아내를 살해한 혐의로 재판을 받고 있습니다. 그는 자신의 무죄를 입증하기 위해 그가 한때 지지를 보낸 고급 AI 판사에게 90분의 시간을 주어야 합니다.
   - **평점**: 7.1
   ![Mercy](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

2. **28 Years Later: The Bone Temple** (2026-01-14)
   - **개요**: Dr. Kelson은 세계를 변화시킬 수 있는 충격적인 새로운 관계에 휘말리게 됩니다.
   - **평점**: 7.2
   ![28 Years Later: The Bone Temple](https://image.tmdb.org/t/p/w780/kK1BGkG3KAvWB0WMV1DfOx9yTMZ.jpg)

3. **Greenland 2: Migration** (2026-01-07)
   - **개요**: 자칫하면 생존이 위태로운 상황 속에서, Garrity 가족은 새로운 집을 찾기 위해 위험한 여정을 떠납니다.
   - **평점**: 6.5
   ![Greenland 2: Migration](https://image.tmdb.org/t/p/w780/z2tqCJLsw6uEJ8nJV8BsQXGa3dr.jpg)

4. **Space/Time** (